# D07 국내 주가지수 시세 API 검증

## 목적
금융위원회 지수시세정보 API를 이용하여 KOSPI·KOSDAQ 일별 지수 데이터를 조회할 수 있는지 검증한다.

## 검증 항목
- KOSPI 조회 가능 여부
- KOSDAQ 조회 가능 여부
- 최근 20거래일 조회
- 주요 시세 필드 확인
- 거래일 중복 여부 확인

> 본 API는 실시간 지수 데이터가 아니라 일별 갱신 데이터이다.

In [ ]:
import requests
import pandas as pd
from getpass import getpass

pd.set_option("display.max_columns", None)

SERVICE_KEY = getpass("공공데이터포털 Decoding 인증키 입력: ")

BASE_URL = (
    "https://apis.data.go.kr/1160100/"
    "GetMarketIndexInfoService_V2/getStockMarketIndex_V2"
)

In [ ]:
def get_index_data(
    service_key,
    index_name,
    begin_date=None,
    end_date=None,
    num_rows=100
):
    params = {
        "serviceKey": service_key,
        "resultType": "json",
        "pageNo": 1,
        "numOfRows": num_rows,
        "idxNm": index_name
    }

    if begin_date is not None:
        params["beginBasDt"] = begin_date

    if end_date is not None:
        params["endBasDt"] = end_date

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=(5, 60)
        )

        print(f"[{index_name}] HTTP 상태코드:", response.status_code)

        if response.status_code != 200:
            print(response.text[:1000])
            return pd.DataFrame()

        data = response.json()

        if "response" not in data:
            print("정상적인 API 응답 구조가 아닙니다.")
            print(data)
            return pd.DataFrame()

        header = data["response"]["header"]

        print(
            "API 결과:",
            header.get("resultCode"),
            header.get("resultMsg")
        )

        body = data["response"].get("body", {})
        items = body.get("items", {})

        if not items:
            print("조회 결과가 없습니다.")
            return pd.DataFrame()

        rows = items.get("item", [])

        if isinstance(rows, dict):
            rows = [rows]

        return pd.DataFrame(rows)

    except requests.exceptions.ReadTimeout:
        print("서버 응답시간 초과")
        return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print("API 요청 오류:", e)
        return pd.DataFrame()

In [ ]:
today = pd.Timestamp.today().normalize()

begin_date = (
    today - pd.Timedelta(days=60)
).strftime("%Y%m%d")

end_date = (
    today + pd.Timedelta(days=1)
).strftime("%Y%m%d")

kospi = get_index_data(
    service_key=SERVICE_KEY,
    index_name="코스피",
    begin_date=begin_date,
    end_date=end_date
)

kosdaq = get_index_data(
    service_key=SERVICE_KEY,
    index_name="코스닥",
    begin_date=begin_date,
    end_date=end_date
)

In [ ]:
def preprocess_index(df):
    if df.empty:
        return df

    df = df.copy()

    df["basDt"] = pd.to_datetime(
        df["basDt"],
        format="%Y%m%d",
        errors="coerce"
    )

    numeric_cols = [
        "clpr",
        "vs",
        "fltRt",
        "mkp",
        "hipr",
        "lopr",
        "trqu",
        "trPrc"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    return df.sort_values(
        "basDt",
        ascending=False
    ).reset_index(drop=True)


kospi = preprocess_index(kospi)
kosdaq = preprocess_index(kosdaq)

In [ ]:
wanted_cols = [
    "basDt",
    "idxNm",
    "clpr",
    "vs",
    "fltRt",
    "mkp",
    "hipr",
    "lopr",
    "trqu"
]

kospi20 = kospi[
    [c for c in wanted_cols if c in kospi.columns]
].head(20).copy()

kosdaq20 = kosdaq[
    [c for c in wanted_cols if c in kosdaq.columns]
].head(20).copy()

rename_cols = {
    "basDt": "날짜",
    "idxNm": "지수명",
    "clpr": "종가",
    "vs": "전일대비",
    "fltRt": "등락률",
    "mkp": "시가",
    "hipr": "고가",
    "lopr": "저가",
    "trqu": "거래량"
}

kospi20 = kospi20.rename(columns=rename_cols)
kosdaq20 = kosdaq20.rename(columns=rename_cols)

print("=== KOSPI 최근 20거래일 ===")
display(kospi20)

print("=== KOSDAQ 최근 20거래일 ===")
display(kosdaq20)

In [ ]:
print("===== D07 검증 결과 =====")

print("KOSPI 데이터 존재:", not kospi20.empty)
print("KOSDAQ 데이터 존재:", not kosdaq20.empty)

print("KOSPI 최근 거래일 수:", len(kospi20))
print("KOSDAQ 최근 거래일 수:", len(kosdaq20))

print(
    "KOSPI 중복 날짜:",
    kospi20["날짜"].duplicated().sum()
)

print(
    "KOSDAQ 중복 날짜:",
    kosdaq20["날짜"].duplicated().sum()
)

## 검증 결과

| 검증 항목 | 결과 |
|---|---|
| API 연결 | 성공 |
| KOSPI 조회 | 성공 |
| KOSDAQ 조회 | 성공 |
| 최근 20거래일 조회 | 성공 |
| 시가·고가·저가·종가 | 확인 |
| 등락률 | 확인 |
| 거래량 | 확인 |
| 거래일 중복 확인 | 가능 |

### 주의사항
- 실시간 데이터가 아닌 일별 갱신 데이터
- 2020-01-01 이후 데이터 제공
- 2024-12-06 이후 일부 지수명 변경 이력이 있어 장기 시계열 연결 시 명칭 기준만 사용하지 않도록 주의
- 공공누리 제4유형 조건 적용